To build a statistically sound Spatio-Temporal Data Cube, every layer must be downsampled into the **`h310grid`** base template. In spatial data science, you cannot simply merge categorical text and numeric densities using the same rule.

The table below organizes the layers by their geometry type and defines the suggested **Spatial Operation** and **Statistical Aggregation** you need to use to prevent data duplication and maintain a perfect 1-to-1 grid matrix.

---

### **Master Grid-Integration Matrix**

| Layer Name | Geometry Type | Recommended Action | Spatial Operation (Python tool) | Statistical Aggregation / Rule |
| --- | --- | --- | --- | --- |
| **`provcm01012026_wgs84`** | Polygon | **DROP** | None | Use this layer at the very beginning to clip your `h310grid` so you only process hexagons inside Florence. |
| **`_waterways`** | Polygon | **DROP** | None | Delete entirely to prevent redundant data bloat. |
| **`com01012026_wgs84`** | Polygon, `[generic location]` | **DROP** | None | Unused since the comunal attributes are created in separate layers|

#### **1. Point Layers (Events & Infrastructure)**

*For infrastructure and events, your goal is to calculate localized density, presence, or financial investment per hexagon.*

| Layer Name | Geometry Type & Variable | Spatial Operation | Statistical Aggregation Rule | Thesis Column Name Idea |
| --- | --- | --- | --- | --- |
| **`info_lotti_multipmpoint`** | Point, `[tipo_disse, importo_lo]` | `gpd.sjoin()` | **`sum`** of the money spent column `importo_lo` for the closest point and type of the closest `tipo_disse`, (from the current hexagon or the 1-ring scale, 1-level neighbors)| `total_intervention_cost`, `class_intervention` |
| **`pubacq_acq_fontanello_aq_attivipoint`** | Point, `[generic location]` | `gpd.sjoin()` | **`count`** of points per hexagon (active fountains), (from the current hexagon or the 1-ring scale, 1-level neighbors) | `active_fountains_count` |
| **`fontanellipoint`** | Point, `[generic location]` | `gpd.sjoin()` | **`count`** of points per hexagon (all fountains), (from the current hexagon or the 1-ring scale, 1-level neighbors) | `total_fountains_count` |
| **`frane_piff_toscana_opendata`** | Point, `[tipo_movimento, autorita_distretto]` | `gpd.sjoin()` | **`count`** of points, or **`mode`** (most common landslide type), (from the current hexagon or the 1-ring scale, 1-level neighbors) | `landslide_point_count`, `dominant_landslide_type` |
| **`_peaks`** | Point, `[ele]` | `[ele] de-string convert to int`, `gpd.sjoin()` | **`max`** of the elevation attribute across the current hexagon or the 1-ring scale, 1-level neighbors | `max_peak_elevation` |
| **`_places`** | Point, `[name]` | `gpd.sjoin()` | **`name`** of the closest name to the hexagon| `is_locality` |
| **`seismic_points_utm32n`** | Point, `[MwDef]` | `gpd.sjoin()` | **`count`** of events, (from the current hexagon or the 1-ring scale, 1-level neighbors), and **`max`** and **`mean`** of the magnitude (`MwDef`) field. | `seismic_event_count`, `max_seismic_magnitude`, `avg_seismic_magnitude` |

#### **2. Point Layers (Environmental Monitoring Stations)**

*Because stations are scattered far apart, standard joining leaves most hexagons empty ($NaN$). You must treat these as continuous surfaces. (check both Interpolation and NN distance for all stations, read about it)*

| Layer Name | Geometry Type & Variable | Spatial Operation / Approach | Statistical Aggregation Rule |
| --- | --- | --- | --- |
| **`termometri_stations_firenze`** | Point, `[quota,temp_max_c,temp_min_c]` | **DROP** | Assign the monthly temperature time-series to the hexagon from its nearest matching station. quota is the altitude of the station |
| **`idrometri_stations_firenze`** | Point, `[quota,water_level_m]` | **DROP** | Calculate `distance_to_nearest_hydrometer` (meters) for risk proximity. quota is the altitude of the station |
| **`cf_pluviometri`** | Point, `[quota,rain_mm]` | **DROP** | Sample the continuous monthly rainfall value directly into each hexagon centroid. quota is the altitude of the station |
| **`anemometri_stations_firenze`** | Point, `[quota,wind_speed_max_m_s,wind_speed_mean_m_s]` | **DROP**  | Assign wind vector speeds from the proxy station network. quota is the altitude of the station |
| **`termometri_stations_firenze_aggregated`** | Point, `[quota, temp_max_peak, temp_min_nadir, temp_thermal_range]`                                                   | Spatial Interpolation (IDW/Kriging) or nearest-neighbor join. |  Assign the monthly temperature time-series to the hexagon from its nearest matching station. quota is the altitude of the station |
| **`idrometri_stations_firenze_aggregated`**  | Point, `[quota, river_stage_max_m, river_stage_mean_m, river_stage_std_m]`                                            | Nearest-neighbor distance calculation (`gpd.sjoin_nearest`).  | Calculate `distance_to_nearest_hydrometer` (meters) for risk proximity. quota is the altitude of the station |
| **`cf_pluviometri_aggregated`**            | Point, `[quota, rain_mm_sum_annual, rain_mm_max_monthly, rain_mm_min_monthly, rain_mm_avg_monthly, rain_mm_std]` | Spatial Interpolation (IDW/Kriging) to create continuous surfaces. | Sample the continuous monthly rainfall value directly into each hexagon centroid. quota is the altitude of the station |
| **`anemometri_stations_firenze_aggregated`** | Point, `[quota, wind_speed_max_max, wind_speed_max_95p, wind_speed_avg_mean]`                                         | Spatial Interpolation (IDW/Kriging) or nearest-neighbor join. | Assign wind vector speeds from the proxy station network. quota is the altitude of the station |


#### **3. Line Layers (Linear Features & Networks)**

*Lines must be converted into continuous geometric metrics (density or distance) so they can fit into a polygon grid cell.*

| Layer Name | Geometry Type & Variable | Recommended Action | Spatial Operation | Statistical Aggregation Rule |
| --- | --- | --- | --- | --- |
| **`_road`** | Line, `[highway, surface, tunnel, bridge]` | **KEEP** | `gpd.overlay(how='intersection')` | Calculate the **total length** ($m$) of roads inside each hexagon divided by the hexagon area to get road density ($m/m^2$). keep the average value for  `[highway, surface, tunnel, bridge]`|
| **`pciv_tombinatiline_f`** | Line, `[drop layer]` | **DROP** (Keep if urban flood risk is a factor) but is very similar to `waterwaysL`. | `gpd.overlay(how='intersection')` or proximity. | Calculate total length per hexagon. If it only covers Florence city center and your grid is provincial, fill outer areas with `0`. |
| **`corso_fi`** | Line, `[drop layer]` | **DROP** (Critical for hydraulic risk) but is very similar to `waterwaysL`. | `gpd.GeoSeries.distance()` | Calculate the exact distance from each hexagon's **centroid** to the nearest waterway line segment. |
| **`waterwaysL`** | Line, `[waterway, name]` | **KEEP** (Critical for hydraulic risk) | `gpd.GeoSeries.distance()` | Calculate the exact distance from each hexagon's **centroid** to the nearest waterway line segment, and create a categorical measurement of which types of water courses pass through the hexagon. |

#### **4. Polygon Layers (Soil, Hazards, Land Use, & Administration)**

*When a hexagon overlaps categorical or ordinal soil polygons, you must select a single attribute using a majority rule to keep the grid flat and clean. (deagregate the categories  aswell to binary put all present variables, or sum or count all present variables)*

| Layer Name | Geometry Type & Variable | Spatial Operation | Statistical Aggregation Rule (How to collapse to 1 row) |
| --- | --- | --- | --- |
| **`profondita_utile_per_le_radici_cm`** | Polygon, `[profond]` | `gpd.overlay(how='intersection')` | **Majority Area Rule:** Majority class Calculate which rooting depth class covers the highest percentage area of the hexagon. |
| **`pietrosita_superficiale_`** | Polygon, `[ciottoli]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class (Absent to Abundant). |
| **`natural_`** | Polygon, `[natural]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** and the first 2 most present classes, to extract dominant natural classification label. |
| **`mph_mosaicatura_ispra_2020_pericolosita_idraulica_firenze`** | Polygon, `[generic location]` | **DROP** | combined to `aggr_mosaicatura_ispra_2020_pericolosita_idraulica_firenze` |
| **`lph_mosaicatura_ispra_2020_pericolosita_idraulica_firenze`** | Polygon, `[generic location]` | **DROP** | combined to `aggr_mosaicatura_ispra_2020_pericolosita_idraulica_firenze` |
| **`hph_mosaicatura_ispra_2020_pericolosita_idraulica_firenze`** | Polygon, `[generic location]` | **DROP** | combined to `aggr_mosaicatura_ispra_2020_pericolosita_idraulica_firenze` |
| **`aggr_mosaicatura_ispra_2020_pericolosita_idraulica_firenze`** | Polygon, `[pericolo]` | `gpd.overlay(how='intersection')` or Spatial Join | **Ordinal Max Priority Rule:** Majority class, For each hexagon, extract all intersecting hazard values (`1`=low, `2`=medium, `3`=high) and assign the maximum value present ($\max(\text{values})$). If a hexagon contains no overlapping hazard polygons, assign a default baseline value of `0`. |
| **`ksat_30__conducibilita_idraulica_satura_sezione_030_cm`** | Polygon, `[ksat_30]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Convert Ordinal text to numeric 1–5 ranking). |
| **`ksat_150__conducibilita_idraulica_satura_sezione_0150_cm`** | Polygon, `[ksat_150]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Convert Ordinal text to numeric 1–5 ranking). |
| **`interferenza_climatica_per_quota`** | Polygon, `[interf_cli]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Absent to Extreme). |
| **`interferenza_climatica_per_deficit_idrico`** | Polygon, `[deficit]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Absent to Very Strong). |
| **`gruppo_idrologico_usda`** | Polygon, `[gi]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, to assign the single dominant letter group (A, B, C, or D). |
| **`franosita__di_superficie_interessata_da_frane`** | Polygon, `[franosita]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class |
| **`frane_poly_toscana_opendata`** | Polygon, `[generic location]` | `gpd.overlay(how='intersection')` | Calculate total area ($m^2$) of the hexagon covered by active polygon landslides. |
| **`velocita_movimento_nel_piano`** | Polygon, `[v_eozn, ang_eozn]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** for the movement directional vector/label. |
| **`celle_soli_PS_descendenti`** | Polygon, `[ave_vdesc]` | `gpd.sjoin()` | **Binary Flag:** `1` if hexagon intersects a descending soil movement cell, `0` if not. and average speed |
| **`celle_soli_PS_ascendenti`** | Polygon, `[ave_vasc]` | `gpd.sjoin()` | **Binary Flag:** `1` if hexagon intersects an ascending soil movement cell, `0` if not. and average speed |
| **`forest_`** | Polygon, `[landuse]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, to get the dominant land-use category string. |
| **`unita_di_paesaggio`** | Polygon, `[udp, udp_descri]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** for landscape unit label (`udp`). keep a column for each of the 3 most prevalent  udp values in the hexagon|
| **`sottosistemi_di_paesaggio`** | Polygon, `[sst, sst_descri]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** for broad landscape label (`sst`). keep a column for each of the 3 most prevalent  sst values in the hexagon|
| **`sistemi_di_paesaggio`** | Polygon, `[sg, sg_descri]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** for detailed landscape label (`sg`). keep a column for each of the 3 most prevalent  sg values in the hexagon|
| **`soil_region`** | Polygon, `[srg, srg_descri]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** for general soil region classification (`srg`). keep a column for each of the 3 most prevalent  srg values in the hexagon|
| **`fertilita_chimica_dellorizzonte_superficiale`** | Polygon, `[fertilita]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Low to Good). |
| **`erosione_potenziale_tha`** | Polygon, `[erosione]` | `gpd.overlay(how='intersection')` | **Majority Area Rule**  Majority class, (Absent to Very High continuous scale). |
| **`drenaggio_interno`** | Polygon, `[drenag]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Very bad to Good). |
| **`capacita_duso_e_fertilita_dei_suoli`** | Polygon, `[lcc_classe]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Extract the dominant LCC class 1–8). |
| **`building_`** | Polygon, `[building, amenity]` | `gpd.overlay(how='intersection')` | Calculate **Total Built Area Footprint** ($m^2$) inside the hexagon, or count individual building centroids. Also assign all categories included in each hex for the variable building |
| **`salinita_dellorizzonte_superficiale_mscm__125`** | Polygon, `[salinita]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Topsoil salinity class). |
| **`salinita_dellorizzonte_sottosuperficiale_1m_mscm__125`** | Polygon, `[sal_prof]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Subsoil salinity class). |
| **`rocciosita_`** | Polygon, `[rocciosita]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Absent to Extremely Rocky). |
| **`rischio_di_inondazione_con_tempo_di_ritorno_inferiore_a_30_anni`** | Polygon, `[inondaz]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Absent to Extremely Elevated). |
| **`awc__available_water_capacity`** | Polygon, `[awc]` | `gpd.overlay(how='intersection')` | **Majority Area Rule** Majority class, (Very low to Very high values). |
| **`tabular_comunal_census`** | Polygon, `[census_pop]` | `gpd.sjoin(predicate='within')` on H3 Centroids | **Centroid Structural Assignment:** Assign the municipal population value to all contained hexagon centroids to act as a baseline per-capita denominator. |
| **`tabular_comunal_economical_princ`** | Polygon, `[NTAXP, TAXABINCR, CADINCR, CADINCF, SUBEMPTRINCR, SUBEMPTRINCF, PENSINCR, PENSINCF, ENTROAINCR, ENTROAINCF]` | `gpd.sjoin(predicate='within')` on H3 Centroids | **Centroid Structural Assignment:** Transfer total tax volume scales and absolute income categories based on which municipality the hexagon center point occupies. Used to generate localized structural proxies ($\frac{\text{TAXABINCR}}{\text{NTAXP}}$). |
| **`tabular_comunal_economical_reddito`** | Polygon, `[E0-10000, E10000-15000, E15000-26000, E26000-55000, E55000-75000]` | `gpd.sjoin(predicate='within')` on H3 Centroids | **Centroid Structural Assignment:** Assign bracket counts directly to the local cell level to calculate financial stratification indices and vulnerable economic densities. assign based on which municipality the hexagon center point occupies. |
| **`tabular_comunal_economical_distrib`** | Polygon, `[ACQ_IMM, ACQ_EROG]` | `gpd.sjoin(predicate='within')` on H3 Centroids | **Centroid Structural Assignment:** Map volumes directly to compute network mass balance ($\text{ACQ\_IMM} - \text{ACQ\_EROG}$) to isolate physical water loss metrics across the cell territory. assign based on which municipality the hexagon center point occupies.|
| **`tabular_comunal_economical_indice_comp`** | Polygon, `[COMP_FRAG_INDEX_DECILE, LAND_CONSUMPTION, EMPL_RATE_20_64, POP_25_64_UNDER_DIP_LOW_SEC_EDU, POP_DEPEN_INDEX_ADJ, INDEX_ACCES_ESSENT_SERVICES, PERSEMP_LU_LOW_PRO_INDSERV_VENTILE]` | `gpd.sjoin(predicate='within')` on H3 Centroids | **Centroid Structural Assignment / Filtered Transfer:** Map socio-demographic indicators, education gaps, and job index variables directly. assign based on which municipality the hexagon center point occupies. |
| **`pendolarismo_inflow_comunal`** | Polygon, `[inflow_total]` | `gpd.sjoin()` via H3 Centroids & proportional building area allocation | Map demografic values for each hexagon depending on the municipality is in,   assign based on which municipality the hexagon center point occupies.|
| **`h3_grid_with_dtm_stats`** | Polygon, `[dtmidcnt_mean, dtmidcnt_max]` | Direct Attribute Merge (Native Grid) | **Direct Join / Native Variable:** Direct 1-to-1 attribute join via `index`. No additional spatial operation or aggregation required, as zonal raster statistics were pre-calculated directly on the H3 cell geometry. |
| **`mosaicatura_ispra_2024_pericolosita_frana_pai`** | Polygon, `[per_fr_ita]` | `gpd.overlay(how='intersection')` or Spatial Join | **Ordinal Max Priority Rule:** Consolidate categorical strings into a single ranked scale (`0` = None, `1` = Attention Zone AA, `2` = Moderate P1, `3` = Medium P2, `4` = High P3, `5` = Very High P4). Assign the maximum value present within the hexagon boundary ($\max(\text{values})$). If no polygon intersects, default the cell value to `0`. |
---

### **Implementation Rule of Thumb**

When building the loop in Python, process the vector data layers using a **two-tiered function strategy**:

1. **For Ordinal/Categorical Polygons:** Use a helper function that cuts the layers by intersection (`gpd.overlay`), groups them by `h3_index`, calculates the geometry area of the fragments, and selects the row with the maximum area (`.idxmax()`).
2. **For Counts/Densities (Points/Lines):** Use standard spatial joins (`gpd.sjoin`) and compute aggregate statistics (`.groupby('h3_index').agg({'metric': 'sum'})`).